# 04 — Fine-tuning + L1

**L1 = the same extraction pipeline as L0, but driven by a *fine-tuned* EasyOCR
recognizer instead of the stock pretrained one.** Allow-lists, empty-cell skip,
comb assembly and the nb-02 checkbox reads are all unchanged — only the OCR model
improves.

Level scheme (current):
- **L0** — pretrained EasyOCR + allow-lists  (notebook 03)
- **L1** — fine-tuned EasyOCR + allow-lists  (this notebook)
- L2 — L1 + vocab fuzzy snap   (later)
- L3 — L2 + patient-db match    (later)

Data:
- The fine-tuning training set is built in **notebook 02** (`data/train_recog/`,
  the materialized x4-augmented crops for the **80 train** forms only).
- The **80/20 split** lives in `data/generated/split_train_val.json` (written by
  nb 02) so both notebooks agree on which forms are held out.

Train-once design: the training cell writes `models/easyocr_ft/recognizer.pth`
and is **guarded** — if that checkpoint exists it is skipped, so you can re-run
every other cell (inference, eval) without retraining. Set `FORCE_RETRAIN=True`
to retrain.

> No CUDA on this machine, so training runs on CPU — do the one training pass
> overnight. `SMOKE=True` does a tiny dry-run first to confirm the loop works.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import ocr        # src/ocr.py  (L0/L1 extraction pipeline)
import finetune   # src/finetune.py  (CTC fine-tune)

CROPS_ROOT  = ROOT / "data" / "scans" / "crops"
FIELD_MAP   = ROOT / "data" / "template" / "field_map.json"
REPORT      = ROOT / "data" / "scans" / "preprocess_report.json"
LABELS      = ROOT / "data" / "scans" / "labels.csv"
SPLIT       = ROOT / "data" / "generated" / "split_train_val.json"
TRAIN_RECOG = ROOT / "data" / "train_recog"
PRED_L0     = ROOT / "data" / "generated" / "predictions_l0.csv"
PRED_L1     = ROOT / "data" / "generated" / "predictions_l1.csv"
CKPT        = ROOT / "models" / "easyocr_ft" / "recognizer.pth"

field_map = ocr.load_field_map(str(FIELD_MAP))
report    = ocr.load_checkbox_report(str(REPORT))
labels    = pd.read_csv(LABELS, dtype=str).fillna("")
label_cols = [c for c in labels.columns if c != "image"]

# The split + materialized training set must come from nb 02.
assert SPLIT.exists(), "Run the fine-tuning section of notebook 02 first (writes split_train_val.json)."
assert (TRAIN_RECOG / "labels.csv").exists(), "Run nb 02's fine-tuning section first (builds data/train_recog/)."
train_pids, val_pids = finetune.load_split(str(SPLIT))
print(f"train {len(train_pids)} / val {len(val_pids)} forms")
print("train_recog samples:", sum(1 for _ in open(TRAIN_RECOG / "labels.csv")) - 1)

In [ ]:
%%time
# --- Fine-tune (guarded). Heavy: run once, ideally overnight on CPU. ---------
FORCE_RETRAIN = False
SMOKE         = False   # True -> 1 epoch / 5 batches, just to verify the loop

if CKPT.exists() and not FORCE_RETRAIN:
    print(f"checkpoint exists -> skipping training: {CKPT}")
    print("(set FORCE_RETRAIN=True to retrain)")
else:
    stats = finetune.train_recognizer(
        out_dir=str(TRAIN_RECOG),
        ckpt_path=str(CKPT),
        epochs=20,          # lower if overnight is too long
        lr=1e-4,
        batch_size=32,
        gpu=False,
        smoke=SMOKE,
    )
    print(stats)

In [ ]:
# --- Load the fine-tuned reader (L1) and predict the VALIDATION forms. -------
assert CKPT.exists(), "No checkpoint yet — run the training cell above."
reader_ft = ocr.get_finetuned_reader(str(CKPT), gpu=False)

rows = ocr.predict_all(str(CROPS_ROOT), field_map, report,
                       reader=reader_ft, pids=val_pids, progress=True)

pred_l1 = pd.DataFrame(rows)
for c in label_cols:
    if c not in pred_l1.columns:
        pred_l1[c] = ""
pred_l1 = pred_l1[label_cols].fillna("")
PRED_L1.parent.mkdir(parents=True, exist_ok=True)
pred_l1.to_csv(PRED_L1, index=False)
print("wrote", PRED_L1, "(val forms only)")
pred_l1.head()

## L0 vs L1 on the held-out 20 forms

Same 20 validation forms for both, so the comparison is apples-to-apples. L0
numbers are read back from `predictions_l0.csv` (nb 03) and filtered to the val
split — no recompute, and the L0 model never saw these as "training" because L0
isn't trained at all. The lift here is purely the fine-tuned recognizer.

In [ ]:
def norm(s):
    return " ".join(str(s).strip().split()).lower()

assert PRED_L0.exists(), "Run notebook 03 first to produce predictions_l0.csv."
pred_l0 = pd.read_csv(PRED_L0, dtype=str).fillna("")

gt  = labels.set_index("patient_id")
l0  = pred_l0.set_index("patient_id")
l1  = pred_l1.set_index("patient_id")
fields = [c for c in label_cols if c != "patient_id"]
val = [p for p in val_pids if p in l0.index and p in l1.index]

def acc_on(pred_df, field):
    return sum(norm(gt.loc[p, field]) == norm(pred_df.loc[p, field]) for p in val) / len(val)

cmp = pd.DataFrame({
    "L0_%": {f: 100 * acc_on(l0, f) for f in fields},
    "L1_%": {f: 100 * acc_on(l1, f) for f in fields},
})
cmp["delta"] = (cmp["L1_%"] - cmp["L0_%"]).round(1)
cmp = cmp.round(1).sort_values("delta", ascending=False)
print(f"Per-field exact-match on {len(val)} held-out forms:\n")
print(cmp.to_string())

def micro(pred_df):
    hit = sum(norm(gt.loc[p, f]) == norm(pred_df.loc[p, f]) for p in val for f in fields)
    return 100 * hit / (len(val) * len(fields))
print(f"\nMACRO  L0 {cmp['L0_%'].mean():.1f}%  ->  L1 {cmp['L1_%'].mean():.1f}%")
print(f"MICRO  L0 {micro(l0):.1f}%  ->  L1 {micro(l1):.1f}%")

In [ ]:
# Where L1 still misses on the val set (eyeball remaining failure modes).
import itertools
for f in fields:
    misses = [(p, gt.loc[p, f], l1.loc[p, f]) for p in val
              if norm(gt.loc[p, f]) != norm(l1.loc[p, f])]
    if misses:
        print(f"\n[{f}] {len(misses)}/{len(val)} miss (gt -> L1):")
        for p, g, pr in itertools.islice(misses, 3):
            print(f"   {p}: {g!r} -> {pr!r}")